In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.float_format", "{:,.1f}".format)
pd.set_option("display.max_columns", 30)
CARTELLA_PULITI = Path("../dati_puliti")

# dtype=str sul codice: "015" deve restare "015", non diventare 15
leggi = lambda nome: pd.read_csv(CARTELLA_PULITI / nome, dtype={"codice_provincia": str})
istat = leggi("istat_province.csv")
mef = leggi("mef_redditi_province.csv")
omi = leggi("omi_prezzi_province.csv")

for nome, df in [("Istat", istat), ("MEF", mef), ("OMI", omi)]:
    print(f"{nome:<6} {df.shape[0]:>4} righe  {df.shape[1]:>3} colonne")

Istat   108 righe    6 colonne
MEF      12 righe    9 colonne
OMI      12 righe    8 colonne


In [2]:
disocc_2025 = istat[istat["anno"] == 2025].set_index("codice_provincia")["tasso_disoccupazione"]
disocc_media = (istat[istat["anno"].between(2023, 2025)]
                .groupby("codice_provincia")["tasso_disoccupazione"].mean())
pop_2025 = istat[istat["anno"] == 2025].set_index("codice_provincia")["popolazione"]
pop_2019 = istat[istat["anno"] == 2019].set_index("codice_provincia")["popolazione"]

istat_sintesi = pd.DataFrame({
    "disoccupazione_2025": disocc_2025,
    "disoccupazione_media_2023_2025": disocc_media.round(1),
    "popolazione_2025": pop_2025,
    "var_popolazione_2019_2025_pct": ((pop_2025 / pop_2019 - 1) * 100).round(2),
}).reset_index()

# nome e regione da aggiungere
anagrafica = istat[["codice_provincia", "provincia", "regione"]].drop_duplicates()
istat_sintesi = anagrafica.merge(istat_sintesi, on="codice_provincia")
istat_sintesi

,codice_provincia,provincia,regione,disoccupazione_2025,disoccupazione_media_2023_2025,popolazione_2025,var_popolazione_2019_2025_pct
0,015,Milano,Lombardia,3.0,4.1,"3,246,455.0",-0.1
1,016,Bergamo,Lombardia,1.3,1.9,"1,114,780.0",0.7
2,018,Pavia,Lombardia,4.1,4.1,"541,949.0",0.0
3,058,Roma,Lazio,5.3,5.9,"4,222,648.0",-1.0
4,059,Latina,Lazio,5.9,8.1,"566,698.0",0.6
5,060,Frosinone,Lazio,5.3,7.2,"462,661.0",-3.8
6,061,Caserta,Campania,11.3,11.5,"908,451.0",-0.8
7,063,Napoli,Campania,17.9,19.6,"2,963,633.0",-2.8
8,065,Salerno,Campania,9.8,12.4,"1,055,372.0",-2.9
9,071,Foggia,Puglia,13.7,15.8,"590,263.0",-3.5


In [3]:
tab = (istat_sintesi
       .merge(mef[["codice_provincia", "reddito_medio", "quota_sotto_15k", "contribuenti"]],
              on="codice_provincia", how="inner", validate="one_to_one")
       .merge(omi.drop(columns=["provincia"]),
              on="codice_provincia", how="inner", validate="one_to_one"))

assert len(tab) == 12, f"Attese 12 province, trovate {len(tab)}"
print("Unione riuscita ✅ – province:", len(tab), "| colonne:", tab.shape[1])
tab.head(3)

Unione riuscita ✅ – province: 12 | colonne: 16


,codice_provincia,provincia,regione,disoccupazione_2025,disoccupazione_media_2023_2025,popolazione_2025,var_popolazione_2019_2025_pct,reddito_medio,quota_sotto_15k,contribuenti,n_comuni_omi,prezzo_mq_medio,affitto_mq_mese_medio,prezzo_mq_capoluogo,affitto_mq_mese_capoluogo,semestre
0,015,Milano,Lombardia,3.0,4.1,"3,246,455.0",-0.1,"34,803.0",26.7,2486066,133,"3,181.5",11.4,"4,953.1",16.4,2026/1
1,016,Bergamo,Lombardia,1.3,1.9,"1,114,780.0",0.7,"27,907.0",27.3,844716,243,"1,306.8",5.8,"2,043.2",8.8,2026/1
2,018,Pavia,Lombardia,4.1,4.1,"541,949.0",0.0,"27,192.0",27.6,411173,184,"1,090.1",4.2,"1,828.6",6.5,2026/1


In [4]:
MQ_CASA = 80  # superficie della casa standard (ipotesi di progetto)

tab["prezzo_casa"] = tab["prezzo_mq_medio"] * MQ_CASA
tab["prezzo_casa_capoluogo"] = tab["prezzo_mq_capoluogo"] * MQ_CASA
tab["anni_stipendio"] = (tab["prezzo_casa"] / tab["reddito_medio"]).round(1)
tab["anni_stipendio_capoluogo"] = (tab["prezzo_casa_capoluogo"] / tab["reddito_medio"]).round(1)

tab["affitto_mensile"] = (tab["affitto_mq_mese_medio"] * MQ_CASA).round(0)
tab["affitto_su_reddito_pct"] = (tab["affitto_mensile"] / (tab["reddito_medio"] / 12) * 100).round(1)
tab["rendimento_lordo_pct"] = (tab["affitto_mq_mese_medio"] * 12 / tab["prezzo_mq_medio"] * 100).round(2)

tab[["provincia", "regione", "prezzo_casa", "reddito_medio", "anni_stipendio",
     "anni_stipendio_capoluogo", "affitto_su_reddito_pct", "rendimento_lordo_pct"]
   ].sort_values("anni_stipendio")

,provincia,regione,prezzo_casa,reddito_medio,anni_stipendio,anni_stipendio_capoluogo,affitto_su_reddito_pct,rendimento_lordo_pct
5,Frosinone,Lazio,"48,768.0","21,992.0",2.2,3.3,17.0,7.7
11,Lecce,Puglia,"60,928.0","19,688.0",3.1,4.6,15.6,5.0
2,Pavia,Lombardia,"87,208.0","27,192.0",3.2,5.4,14.8,4.6
6,Caserta,Campania,"70,656.0","20,525.0",3.4,4.8,13.1,3.8
1,Bergamo,Lombardia,"104,544.0","27,907.0",3.7,5.9,20.0,5.3
9,Foggia,Puglia,"79,992.0","18,749.0",4.3,5.1,20.5,4.8
4,Latina,Lazio,"106,304.0","21,971.0",4.8,5.3,26.2,5.4
8,Salerno,Campania,"97,496.0","20,517.0",4.8,7.3,16.4,3.5
10,Bari,Puglia,"112,024.0","22,297.0",5.0,6.4,21.1,4.2
3,Roma,Lazio,"205,352.0","29,710.0",6.9,8.3,33.9,4.9


In [5]:
def scala_0_100(serie, meglio_alto=True):
    """Normalizzazione min-max: il migliore vale 100, il peggiore 0."""
    if serie.max() == serie.min():  # tutti uguali: nessuna differenza da premiare
        return pd.Series(50.0, index=serie.index)
    s = (serie - serie.min()) / (serie.max() - serie.min()) * 100
    return s if meglio_alto else 100 - s

PESI = {"accessibilita": 0.50, "lavoro": 0.25, "vitalita": 0.25}

tab["score_accessibilita"] = scala_0_100(tab["anni_stipendio"], meglio_alto=False)
tab["score_lavoro"] = scala_0_100(tab["disoccupazione_media_2023_2025"], meglio_alto=False)
tab["score_vitalita"] = scala_0_100(tab["var_popolazione_2019_2025_pct"], meglio_alto=True)

tab["punteggio_convenienza"] = (
    tab["score_accessibilita"] * PESI["accessibilita"]
    + tab["score_lavoro"] * PESI["lavoro"]
    + tab["score_vitalita"] * PESI["vitalita"]
).round(1)
tab["classifica"] = tab["punteggio_convenienza"].rank(ascending=False, method="min").astype(int)

tab[["classifica", "provincia", "regione", "anni_stipendio", "disoccupazione_media_2023_2025",
     "var_popolazione_2019_2025_pct", "punteggio_convenienza"]].sort_values("classifica")

,classifica,provincia,regione,anni_stipendio,disoccupazione_media_2023_2025,var_popolazione_2019_2025_pct,punteggio_convenienza
1,1,Bergamo,Lombardia,3.7,1.9,0.7,85.3
2,2,Pavia,Lombardia,3.2,4.1,0.0,83.5
5,3,Frosinone,Lazio,2.2,7.2,-3.8,67.5
6,4,Caserta,Campania,3.4,11.5,-0.8,66.5
4,5,Latina,Lazio,4.8,8.1,0.6,65.3
11,6,Lecce,Puglia,3.1,10.3,-2.8,60.0
10,7,Bari,Puglia,5.0,6.7,-1.2,55.0
0,8,Milano,Lombardia,7.3,4.1,-0.1,42.4
8,9,Salerno,Campania,4.8,12.4,-2.9,39.7
3,10,Roma,Lazio,6.9,5.9,-1.0,39.1


In [6]:
ordine = [
    "classifica", "codice_provincia", "provincia", "regione",
    "punteggio_convenienza", "anni_stipendio", "anni_stipendio_capoluogo",
    "prezzo_mq_medio", "prezzo_mq_capoluogo", "prezzo_casa", "prezzo_casa_capoluogo",
    "affitto_mq_mese_medio", "affitto_mq_mese_capoluogo", "affitto_mensile",
    "affitto_su_reddito_pct", "rendimento_lordo_pct",
    "reddito_medio", "quota_sotto_15k", "contribuenti",
    "disoccupazione_2025", "disoccupazione_media_2023_2025",
    "popolazione_2025", "var_popolazione_2019_2025_pct",
    "score_accessibilita", "score_lavoro", "score_vitalita",
    "n_comuni_omi", "semestre",
]
finale = tab[ordine].sort_values("classifica").round(2)
finale.to_csv(CARTELLA_PULITI / "tabella_finale_province.csv", index=False)

fonti = pd.DataFrame({
    "Dato": ["Prezzi e affitti al m²", "Reddito medio", "Disoccupazione", "Popolazione", "Ipotesi"],
    "Fonte": ["Agenzia Entrate - OMI, 1° semestre 2026 (abitazioni civili, stato normale)",
              "MEF - Dipartimento delle Finanze, anno d'imposta 2024 (reddito complessivo per contribuente)",
              "Istat, tasso di disoccupazione 15-74 anni (dataflow 151_914_DF_DCCV_TAXDISOCCU1_8)",
              "Istat, popolazione residente al 1° gennaio (dataflow 22_289)",
              f"Casa standard di {MQ_CASA} m²; pesi del punteggio: {PESI}"],
})
with pd.ExcelWriter(CARTELLA_PULITI / "tabella_finale_province.xlsx") as writer:
    finale.to_excel(writer, sheet_name="Province", index=False)
    fonti.to_excel(writer, sheet_name="Fonti", index=False)

print("Salvati in dati_puliti:")
print(" - tabella_finale_province.csv")
print(" - tabella_finale_province.xlsx")

Salvati in dati_puliti:
 - tabella_finale_province.csv
 - tabella_finale_province.xlsx
